# Pattern Backtest

## 目的
- 基于 `patterns.parquet`、`screen_aggregate.parquet` 和 `returns.parquet` 做真实 benchmark 内的技术分析回测
- 支持评分排序、market weighted、可选行业中性
- 支持 Plotly 回测图、个股 pattern 时间点图、事件后价格演化图和选股原因可视化

## 使用方式
1. 先设置“路径配置”和“策略配置”单元
2. 再运行“执行回测”单元
3. 最后运行“可视化”单元


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

from pattern_backtest_engine import (
    get_selection_reason,
    load_backtest_data,
    run_ranked_pattern_backtest,
)
from pattern_backtest_viz import (
    build_event_study_frame,
    make_backtest_figure,
    make_company_pattern_figure,
    make_event_study_figure,
    make_selection_reason_figure,
)

project_dir = Path.cwd()
_TP_ROOT = next((p for p in [project_dir, *project_dir.parents] if (p / "tp_core").exists()), Path(r"C:\GoogleDrive\TP"))
if str(_TP_ROOT) not in sys.path:
    sys.path.insert(0, str(_TP_ROOT))
from tp_core.data_sources import RETURNS_PATH as CANONICAL_RETURNS_PATH
from tp_core.data_sources import SCREEN_AGGREGATE_PATH

patterns_path = Path(os.environ.get('TA_PATTERNS_PATH', str(project_dir / 'output' / 'patterns.parquet')))
returns_path = Path(os.environ.get('TA_RETURNS_PATH', str(CANONICAL_RETURNS_PATH)))
screen_path = Path(os.environ.get('TA_SCREEN_PATH', str(SCREEN_AGGREGATE_PATH)))

for label, path in {
    'patterns_path': patterns_path,
    'returns_path': returns_path,
    'screen_path': screen_path,
}.items():
    if not path.exists():
        raise FileNotFoundError(f'未找到 {label}: {path}')

print(f'patterns_path = {patterns_path}')
print(f'returns_path = {returns_path}')
print(f'screen_path = {screen_path}')


In [2]:
data = load_backtest_data(
    patterns_path=patterns_path,
    returns_path=returns_path,
    screen_path=screen_path,
)

print('patterns shape:', data.patterns.shape)
print('returns shape:', data.returns.shape)
print('screen shape:', data.screen.shape)
print('patterns date range:', data.patterns['Date'].min(), '->', data.patterns['Date'].max())
print('returns date range:', data.returns.index.min(), '->', data.returns.index.max())
print('screen date range:', data.screen['Date'].min(), '->', data.screen['Date'].max())

preview_cols = [
    col
    for col in ['Date', 'triangle_pattern', 'wedge_pattern', 'double_pattern', 'signal', 'rsi_14', 'momentum_10']
    if col in data.patterns.columns
]
display(data.patterns[preview_cols].tail())


patterns shape: (2183748, 98)
returns shape: (5445, 11836)
screen shape: (3421301, 190)
patterns date range: 2005-01-03 00:00:00 -> 2026-03-30 00:00:00
returns date range: 2005-01-03 00:00:00 -> 2026-04-01 00:00:00
screen date range: 1999-12-31 00:00:00 -> 2026-03-31 00:00:00


,Date,triangle_pattern,wedge_pattern,double_pattern,signal,rsi_14,momentum_10
Company SEDOL,,,,,,,
J2N4RC-R,2026-03-30,Ascending Triangle,None,None,LH,58.708771,60.308023
J2MW73-R,2026-03-30,Ascending Triangle,None,None,LL,36.091949,-288.017428
J2MLPV-R,2026-03-30,Ascending Triangle,None,None,None,44.841068,-126.507736
J2HM33-R,2026-03-30,Ascending Triangle,Wedge Up,None,HL,47.696119,-19.992495
XQG0NZ-R,2026-03-30,Ascending Triangle,Wedge Up,None,None,60.757607,79.073778


In [10]:
# 候选 score 速查表
# 数值型字段可以直接参与排序；形态型字段需要配合 score_value_map 指定目标值。
# 下面“形态取值含义”解释的是本项目当前源码里的程序化定义，不是严格教材定义。

def sample_pattern_values(column: str, limit: int = 5) -> str:
    if column not in data.patterns.columns:
        return 'N/A'

    series = data.patterns[column].dropna()
    if series.empty:
        return 'N/A'
    if pd.api.types.is_bool_dtype(series):
        return 'True / False'

    values = []
    for value in series.astype(str).unique().tolist():
        value_lower = value.lower()
        if value_lower in {'none', 'nan', 'false', ''}:
            continue
        values.append(value)
    return ', '.join(values[:limit]) if values else 'N/A'


numeric_score_reference = pd.DataFrame(
    [
        {'column': 'rsi_14', 'category': '动量/强弱', 'default_direction': 'True(趋势) / False(均值回归)'},
        {'column': 'rsi_21', 'category': '动量/强弱', 'default_direction': 'True(趋势) / False(均值回归)'},
        {'column': 'momentum_10', 'category': '动量', 'default_direction': 'True'},
        {'column': 'momentum_20', 'category': '动量', 'default_direction': 'True'},
        {'column': 'momentum_50', 'category': '中期趋势', 'default_direction': 'True'},
        {'column': 'MACD_12_26_9', 'category': '趋势', 'default_direction': 'True'},
        {'column': 'MACDh_12_26_9', 'category': '趋势加速度', 'default_direction': 'True'},
        {'column': 'MACD_20_50_18', 'category': '中期趋势', 'default_direction': 'True'},
        {'column': 'rvi_14', 'category': '相对波动强弱', 'default_direction': 'True'},
        {'column': 'atr_14', 'category': '波动', 'default_direction': 'False(偏低波)'},
        {'column': 'stdev_20', 'category': '波动', 'default_direction': 'False(偏低波)'},
        {'column': 'BBP_20_2.0_2.0_20_2.0', 'category': '布林位置', 'default_direction': 'True(突破) / False(回归)'},
    ]
)
numeric_score_reference['available'] = numeric_score_reference['column'].isin(data.patterns.columns)

pattern_score_reference = pd.DataFrame(
    [
        {'column': 'triangle_pattern', 'category': '字符串形态', 'example_score_value': 'Ascending Triangle'},
        {'column': 'wedge_pattern', 'category': '字符串形态', 'example_score_value': 'Wedge Up'},
        {'column': 'double_pattern', 'category': '字符串形态', 'example_score_value': 'Double Bottom'},
        {'column': 'head_shoulder_pattern', 'category': '字符串形态', 'example_score_value': 'Inverse Head and Shoulder'},
        {'column': 'multiple_top_bottom_pattern', 'category': '字符串形态', 'example_score_value': 'Multiple Bottom'},
        {'column': 'channel_pattern', 'category': '字符串形态', 'example_score_value': 'Channel Up'},
        {'column': 'signal', 'category': '价格结构标签', 'example_score_value': "['HH', 'HL']"},
        {'column': 'BullishEngulfing', 'category': '布尔K线形态', 'example_score_value': 'True'},
        {'column': 'MorningStar', 'category': '布尔K线形态', 'example_score_value': 'True'},
        {'column': 'Hammer', 'category': '布尔K线形态', 'example_score_value': 'True'},
    ]
)
pattern_score_reference['available'] = pattern_score_reference['column'].isin(data.patterns.columns)
pattern_score_reference['sample_values'] = pattern_score_reference['column'].map(sample_pattern_values)

pattern_value_meaning_reference = pd.DataFrame(
    [
        {'column': 'triangle_pattern', 'value': 'Ascending Triangle', 'meaning': '高点平台、低点抬高，偏多突破结构', 'bias': '偏多'},
        {'column': 'triangle_pattern', 'value': 'Descending Triangle', 'meaning': '低点平台、高点下移，偏空跌破结构', 'bias': '偏空'},
        {'column': 'wedge_pattern', 'value': 'Wedge Up', 'meaning': '价格上行收敛，常见于上涨后的整理或衰竭', 'bias': '中性偏空'},
        {'column': 'wedge_pattern', 'value': 'Wedge Down', 'meaning': '价格下行收敛，常见于下跌后的整理或衰竭', 'bias': '中性偏多'},
        {'column': 'double_pattern', 'value': 'Double Top', 'meaning': '双顶，二次冲高失败', 'bias': '偏空'},
        {'column': 'double_pattern', 'value': 'Double Bottom', 'meaning': '双底，二次探底企稳', 'bias': '偏多'},
        {'column': 'head_shoulder_pattern', 'value': 'Head and Shoulder', 'meaning': '头肩顶，中间头部高于两侧肩部', 'bias': '偏空'},
        {'column': 'head_shoulder_pattern', 'value': 'Inverse Head and Shoulder', 'meaning': '倒头肩底，中间低点低于两侧肩部', 'bias': '偏多'},
        {'column': 'multiple_top_bottom_pattern', 'value': 'Multiple Top', 'meaning': '多次上冲阻力失败', 'bias': '偏空'},
        {'column': 'multiple_top_bottom_pattern', 'value': 'Multiple Bottom', 'meaning': '多次下探支撑后企稳', 'bias': '偏多'},
        {'column': 'channel_pattern', 'value': 'Channel Up', 'meaning': '价格沿上行通道运行', 'bias': '偏多'},
        {'column': 'channel_pattern', 'value': 'Channel Down', 'meaning': '价格沿下行通道运行', 'bias': '偏空'},
        {'column': 'signal', 'value': 'HH', 'meaning': 'Higher High，更高高点，趋势强化', 'bias': '偏多'},
        {'column': 'signal', 'value': 'HL', 'meaning': 'Higher Low，更高低点，结构转强', 'bias': '偏多'},
        {'column': 'signal', 'value': 'LH', 'meaning': 'Lower High，更低高点，反弹转弱', 'bias': '偏空'},
        {'column': 'signal', 'value': 'LL', 'meaning': 'Lower Low，更低低点，下跌延续', 'bias': '偏空'},
    ]
)

print('数值型候选 score：')
display(numeric_score_reference)

print('形态型候选 score：')
display(pattern_score_reference)

print('形态取值含义：')
display(pattern_value_meaning_reference)

print('示例配置：')
print("score_columns = ['triangle_pattern', 'signal', 'rsi_14', 'momentum_10', 'MACDh_12_26_9']")
print("score_weights = [0.15, 0.15, 0.20, 0.25, 0.25]")
print("higher_is_better = [True, True, True, True, True]")
print("score_value_map = {'triangle_pattern': 'Ascending Triangle', 'signal': ['HH', 'HL']}")


数值型候选 score：


,column,category,default_direction,available
0,rsi_14,动量/强弱,True(趋势) / False(均值回归),True
1,rsi_21,动量/强弱,True(趋势) / False(均值回归),True
2,momentum_10,动量,True,True
3,momentum_20,动量,True,True
4,momentum_50,中期趋势,True,True
5,MACD_12_26_9,趋势,True,True
6,MACDh_12_26_9,趋势加速度,True,True
7,MACD_20_50_18,中期趋势,True,True
8,rvi_14,相对波动强弱,True,True
9,atr_14,波动,False(偏低波),True


形态型候选 score：


,column,category,example_score_value,available,sample_values
0,triangle_pattern,字符串形态,Ascending Triangle,True,"Ascending Triangle, Descending Triangle"
1,wedge_pattern,字符串形态,Wedge Up,True,"Wedge Up, Wedge Down"
2,double_pattern,字符串形态,Double Bottom,True,"Double Bottom, Double Top"
3,head_shoulder_pattern,字符串形态,Inverse Head and Shoulder,True,"Inverse Head and Shoulder, Head and Shoulder"
4,multiple_top_bottom_pattern,字符串形态,Multiple Bottom,True,N/A
5,channel_pattern,字符串形态,Channel Up,True,"Channel Up, Channel Down"
6,signal,价格结构标签,"['HH', 'HL']",True,"LH, LL, HH, HL"
7,BullishEngulfing,布尔K线形态,True,True,True
8,MorningStar,布尔K线形态,True,True,True
9,Hammer,布尔K线形态,True,True,True / False


形态取值含义：


,column,value,meaning,bias
0,triangle_pattern,Ascending Triangle,高点平台、低点抬高，偏多突破结构,偏多
1,triangle_pattern,Descending Triangle,低点平台、高点下移，偏空跌破结构,偏空
2,wedge_pattern,Wedge Up,价格上行收敛，常见于上涨后的整理或衰竭,中性偏空
3,wedge_pattern,Wedge Down,价格下行收敛，常见于下跌后的整理或衰竭,中性偏多
4,double_pattern,Double Top,双顶，二次冲高失败,偏空
5,double_pattern,Double Bottom,双底，二次探底企稳,偏多
6,head_shoulder_pattern,Head and Shoulder,头肩顶，中间头部高于两侧肩部,偏空
7,head_shoulder_pattern,Inverse Head and Shoulder,倒头肩底，中间低点低于两侧肩部,偏多
8,multiple_top_bottom_pattern,Multiple Top,多次上冲阻力失败,偏空
9,multiple_top_bottom_pattern,Multiple Bottom,多次下探支撑后企稳,偏多


示例配置：
score_columns = ['triangle_pattern', 'signal', 'rsi_14', 'momentum_10', 'MACDh_12_26_9']
score_weights = [0.15, 0.15, 0.20, 0.25, 0.25]
higher_is_better = [True, True, True, True, True]
score_value_map = {'triangle_pattern': 'Ascending Triangle', 'signal': ['HH', 'HL']}


In [7]:
# 只改这里，不改下面的执行单元
strategy_name = 'TA Rank Top 30'
benchmark_name = 'SP500'

# 现在可以把“形态列”和“数值指标”一起放进 score_columns。
score_columns = ['triangle_pattern', 'signal', 'rsi_14', 'momentum_10', 'MACDh_12_26_9']
score_weights = [0.15, 0.15, 0.20, 0.25, 0.25]
higher_is_better = [True, True, True, True, True]

# 形态列需要用 score_value_map 指定你想要的目标值。
score_value_map = {
    'triangle_pattern': 'Ascending Triangle',
    'signal': ['HH', 'HL'],
}

top_n = 30
top_pct = None
start_date = '2015-01-01'
min_names = 10

sector_neutral = True
sector_column = ' Benchmark ICB Industry '

show_drawdown = True
top_k_reason = 10

viz_company = None
viz_pattern_columns = ['triangle_pattern', 'wedge_pattern', 'double_pattern', 'signal']
viz_pattern_values = {'triangle_pattern': 'Ascending Triangle'}

viz_event_pattern_column = 'triangle_pattern'
viz_event_pattern_value = 'Ascending Triangle'
viz_event_window = 20
viz_max_events = 100

reason_date = None
reason_company = None

print('benchmark_name =', benchmark_name)
print('score_columns =', score_columns)
print('score_value_map =', score_value_map)
print('sector_neutral =', sector_neutral)


benchmark_name = SP500
score_columns = ['triangle_pattern', 'signal', 'rsi_14', 'momentum_10', 'MACDh_12_26_9']
score_value_map = {'triangle_pattern': 'Ascending Triangle', 'signal': ['HH', 'HL']}
sector_neutral = True


In [8]:
missing_score_columns = [col for col in score_columns if col not in data.patterns.columns]
if missing_score_columns:
    raise KeyError(f'patterns.parquet 缺少评分列: {missing_score_columns}')

result = run_ranked_pattern_backtest(
    data=data,
    benchmark_name=benchmark_name,
    score_columns=score_columns,
    score_weights=score_weights,
    score_value_map=score_value_map,
    higher_is_better=higher_is_better,
    top_n=top_n,
    top_pct=top_pct,
    start_date=start_date,
    strategy_name=strategy_name,
    sector_neutral=sector_neutral,
    sector_column=sector_column,
    min_names=min_names,
)

signal_hits = result.strategy_rebalance.reset_index().groupby('Date').size()
print('再平衡期数:', signal_hits.shape[0])
print('平均持仓数:', round(signal_hits.mean(), 2))
print('最少/最多持仓数:', int(signal_hits.min()), int(signal_hits.max()))

display(result.summary)
display(signal_hits.describe().to_frame(name='count'))
display(result.strategy_rebalance.reset_index().head(10))
display(result.strategy_exec_weights.reset_index().head(10))

reason_date = pd.to_datetime(reason_date) if reason_date is not None else result.strategy_rebalance.reset_index()['Date'].max()
latest_selection = result.strategy_rebalance.reset_index()
latest_selection = latest_selection[latest_selection['Date'] == reason_date].sort_values('Selection Rank')
reason_company = reason_company or latest_selection.iloc[0]['Company SEDOL']

reason = get_selection_reason(
    selection_pool=result.selection_pool,
    company_sedol=reason_company,
    effective_date=reason_date,
    top_k=top_k_reason,
)

print('reason_date =', reason_date)
print('reason_company =', reason_company)
display(reason['row'].to_frame(name='value'))

peer_cols = [
    col for col in ['Company SEDOL', 'Sector', 'Total Score', 'Selection Rank', 'Portfolio weight', 'Benchmark Weight']
    if col in reason['peers'].columns
]
display(reason['peers'][peer_cols].head(top_k_reason))


再平衡期数: 586
平均持仓数: 30.0
最少/最多持仓数: 30 30


,TA Rank Top 30,SP500
start,2015-01-13 00:00:00,2015-01-13 00:00:00
end,2026-04-01 00:00:00,2026-04-01 00:00:00
nav_end,322.675102,408.54535
total_return,2.226751,3.085454
annual_return,0.10806,0.1312
annual_vol,0.193361,0.188903
sharpe_like,0.558854,0.694535
max_drawdown,-0.298967,-0.33629


,count
count,586.0
mean,30.0
std,0.0
min,30.0
25%,30.0
50%,30.0
75%,30.0
max,30.0


,Date,Company SEDOL,Signal Date,Screen Date,Portfolio weight,Sector,Name,Symbol,Total Score,Selection Rank,Sector Rank
0,2015-01-12,B5TG5C-R,2015-01-05,2014-12-31,0.062049,6.0,Express Scripts Holding Company,ESRX,0.832143,19,1
1,2015-01-12,B6ZP8C-R,2015-01-05,2014-12-31,0.029476,5.0,Welltower Inc.,WELL,0.823563,23,3
2,2015-01-12,B81TLL-R,2015-01-05,2014-12-31,0.024763,9.0,Broadcom Inc.,AVGO,0.900000,2,1
3,2015-01-12,BRWKF0-R,2015-01-05,2014-12-31,0.018102,2.0,Tractor Supply Company,TSCO,0.825329,22,3
4,2015-01-12,CC0SQN-R,2015-01-05,2014-12-31,0.021617,10.0,"Level 3 Communications, Inc.",LVLT,0.796429,41,1
5,2015-01-12,DH79H0-R,2015-01-05,2014-12-31,0.027458,3.0,Whirlpool Corporation,WHR,0.863115,8,2
6,2015-01-12,FBCHQC-R,2015-01-05,2014-12-31,0.041993,2.0,"Bath & Body Works, Inc.",BBWI,0.884539,5,1
7,2015-01-12,GC9695-R,2015-01-05,2014-12-31,0.054608,7.0,Sherwin-Williams Company,SHW,0.848000,14,2
8,2015-01-12,HSFL3C-R,2015-01-05,2014-12-31,0.024680,1.0,"PPG Industries, Inc.",PPG,0.859091,9,1
9,2015-01-12,HVQ2KJ-R,2015-01-05,2014-12-31,0.027945,7.0,"Stericycle, Inc.",SRCL,0.800000,38,3


,Date,Company SEDOL,Portfolio weight,Signal Date,Screen Date,Sector,Name,Symbol,Total Score,Selection Rank,Sector Rank
0,2015-01-13,B5TG5C-R,0.062049,2015-01-05,2014-12-31,6.0,Express Scripts Holding Company,ESRX,0.832143,19,1
1,2015-01-13,B6ZP8C-R,0.029476,2015-01-05,2014-12-31,5.0,Welltower Inc.,WELL,0.823563,23,3
2,2015-01-13,B81TLL-R,0.024763,2015-01-05,2014-12-31,9.0,Broadcom Inc.,AVGO,0.900000,2,1
3,2015-01-13,BRWKF0-R,0.018102,2015-01-05,2014-12-31,2.0,Tractor Supply Company,TSCO,0.825329,22,3
4,2015-01-13,CC0SQN-R,0.021617,2015-01-05,2014-12-31,10.0,"Level 3 Communications, Inc.",LVLT,0.796429,41,1
5,2015-01-13,DH79H0-R,0.027458,2015-01-05,2014-12-31,3.0,Whirlpool Corporation,WHR,0.863115,8,2
6,2015-01-13,FBCHQC-R,0.041993,2015-01-05,2014-12-31,2.0,"Bath & Body Works, Inc.",BBWI,0.884539,5,1
7,2015-01-13,GC9695-R,0.054608,2015-01-05,2014-12-31,7.0,Sherwin-Williams Company,SHW,0.848000,14,2
8,2015-01-13,HSFL3C-R,0.024680,2015-01-05,2014-12-31,1.0,"PPG Industries, Inc.",PPG,0.859091,9,1
9,2015-01-13,HVQ2KJ-R,0.027945,2015-01-05,2014-12-31,7.0,"Stericycle, Inc.",SRCL,0.800000,38,3


reason_date = 2026-03-30 00:00:00
reason_company = KV0J41-R


,value
Company SEDOL,KV0J41-R
Signal Date,2026-03-23 00:00:00
triangle_pattern,Ascending Triangle
signal,HH
rsi_14,72.819936
momentum_10,83.515253
MACDh_12_26_9,3.496357
Screen Date,2026-02-28 00:00:00
Benchmark Weight,1.018266
Benchmark Market Value Millions in EUR,505333.7


,Company SEDOL,Sector,Total Score,Selection Rank,Portfolio weight,Benchmark Weight
291646,V9977J-R,8.0,0.954032,1,NaN,0.162418
291280,D1P43P-R,1.0,0.915000,2,NaN,0.026440
291434,KV0J41-R,6.0,0.914091,3,0.075534,1.018266
291570,R2J99W-R,3.0,0.910294,4,0.042251,0.070662
291254,C7C9TP-R,7.0,0.901010,5,0.013897,0.037328
291642,V4RQ7N-R,10.0,0.891667,6,0.021642,0.083838
291495,MT8FDR-R,9.0,0.880000,7,0.020514,0.076002
291601,S7P87W-R,7.0,0.878030,8,0.052997,0.142796
291327,FQYW7Q-R,6.0,0.872727,9,0.000694,0.009414
291338,FXTG1M-R,7.0,0.872475,10,0.033795,0.089997


In [9]:
nav_fig = make_backtest_figure(result, show_drawdown=show_drawdown)
nav_fig.show()

company_to_plot = viz_company or reason_company
company_fig = make_company_pattern_figure(
    patterns=result.data.patterns,
    company_sedol=company_to_plot,
    pattern_columns=viz_pattern_columns,
    pattern_values=viz_pattern_values,
    start_date=start_date,
)
company_fig.show()

event_frame = build_event_study_frame(
    patterns=result.data.patterns,
    returns=result.data.returns,
    pattern_column=viz_event_pattern_column,
    pattern_value=viz_event_pattern_value,
    event_window=viz_event_window,
    max_events=viz_max_events,
    start_date=start_date,
)
display(event_frame.head(10))

event_fig = make_event_study_figure(
    event_frame,
    title=f'{viz_event_pattern_column} = {viz_event_pattern_value} 的事件后价格路径',
)
event_fig.show()

reason_fig = make_selection_reason_figure(
    selection_pool=result.selection_pool,
    company_sedol=reason_company,
    effective_date=reason_date,
    top_k=top_k_reason,
)
reason_fig.show()

company_weekly = result.data.patterns.reset_index().copy()
company_weekly = company_weekly[company_weekly['Company SEDOL'] == company_to_plot].copy()
company_weekly['Date'] = pd.to_datetime(company_weekly['Date'])
window_mask = company_weekly['Date'].between(reason_date - pd.Timedelta(days=90), reason_date + pd.Timedelta(days=90))
detail_cols = [
    col
    for col in ['Date', 'Open', 'High', 'Low', 'Close', *viz_pattern_columns, *score_columns]
    if col in company_weekly.columns
]
display(company_weekly.loc[window_mask, detail_cols].tail(20))


,event_id,horizon,normalized_price,Company SEDOL,Signal Date,Effective Date,Exec Date
0,0,0,1.000000,Q553HS-R,2015-01-05,2015-01-12,2015-01-13
1,0,1,0.984063,Q553HS-R,2015-01-05,2015-01-12,2015-01-13
2,0,2,1.096560,Q553HS-R,2015-01-05,2015-01-12,2015-01-13
3,0,3,1.123185,Q553HS-R,2015-01-05,2015-01-12,2015-01-13
4,0,4,1.087710,Q553HS-R,2015-01-05,2015-01-12,2015-01-13
5,0,5,1.103958,Q553HS-R,2015-01-05,2015-01-12,2015-01-13
6,0,6,1.088959,Q553HS-R,2015-01-05,2015-01-12,2015-01-13
7,0,7,1.093189,Q553HS-R,2015-01-05,2015-01-12,2015-01-13
8,0,8,1.100692,Q553HS-R,2015-01-05,2015-01-12,2015-01-13
9,0,9,1.080657,Q553HS-R,2015-01-05,2015-01-12,2015-01-13


,Date,Open,High,Low,Close,triangle_pattern,wedge_pattern,double_pattern,signal,triangle_pattern,signal,rsi_14,momentum_10,MACDh_12_26_9
2159583,2026-01-05,686.503574,698.083262,686.503574,690.518595,None,None,None,HL,None,HL,67.473441,50.982964,1.518000
2161556,2026-01-12,706.038991,744.296899,706.038991,741.659619,Ascending Triangle,Wedge Up,None,LL,Ascending Triangle,LL,75.830055,111.577094,3.416094
2163530,2026-01-19,738.792394,738.792394,731.739896,736.359665,None,Wedge Up,None,HH,None,HH,73.716355,76.974314,3.780735
2165504,2026-01-26,732.249631,751.260773,732.249631,751.260773,Ascending Triangle,Wedge Up,None,LH,Ascending Triangle,LH,75.761999,58.400624,4.445691
2167480,2026-02-02,769.591444,798.546150,769.591444,798.546150,Ascending Triangle,Wedge Up,None,None,Ascending Triangle,None,80.854250,97.341638,7.320575
2169455,2026-02-09,787.854994,809.234553,786.765479,808.008104,Ascending Triangle,Wedge Up,None,None,Ascending Triangle,None,81.683516,126.210426,8.999406
2171430,2026-02-16,807.667268,825.553088,807.667268,809.089428,Ascending Triangle,Wedge Up,None,None,Ascending Triangle,None,81.780637,100.216963,9.285223
2173406,2026-02-23,819.710609,831.872608,815.851356,831.872608,Ascending Triangle,Wedge Up,None,None,Ascending Triangle,None,83.737268,138.772919,10.046742
2175381,2026-03-02,841.357926,842.917054,817.993823,820.870343,None,Wedge Up,None,None,None,None,79.307872,127.093447,8.887495
2177359,2026-03-09,827.312260,834.387428,827.278383,834.387428,Ascending Triangle,Wedge Up,None,HH,Ascending Triangle,HH,80.661310,140.455607,8.129630
